In [1]:
import pandas as pd
from IPython.display import display
from src.research_config import ResearchConfig
from src.execution import entry_option_terms
from src.execution import budgeted_size
from src.data_helpers import read_series
from src.backtest import black_scholes_price
from src.backtest import option_types_from_spread_direction


# 06 Option Pricing and Expiry Selection

Convert the Module 04 snapshot into synthetic next-close option terms. The pricing and integer-sizing functions are shared with Module 07. This is an illustration at the saved preview date, not a second backtest.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Imports and saved run


In [ ]:
cfg = ResearchConfig().validate()


## 2. Load snapshot and pricing inputs

Expiry is signal session plus H observed sessions. The remaining maturity at next-close entry is H minus one sessions.


In [2]:
snapshot = pd.read_parquet("signal_snapshot.parquet")
test_prices = pd.read_parquet("test_prices.parquet")
volatility = pd.read_parquet("oos_ewma_volatility.parquet")
rates = read_series("data/inputs/risk_free_rates.parquet")
display(snapshot)


,pair,dependent,independent,beta,signal_date,direction,z,horizon,probability
0,ABT-ZTS,ABT,ZTS,0.841182,2022-12-28,1,2.067096,84,0.7050
1,CDW-MSI,CDW,MSI,1.126032,2022-12-28,-1,-1.629882,91,0.7044
2,HD-SHW,HD,SHW,0.904478,2022-12-28,1,2.350245,62,0.7038
3,LEN-MAS,LEN,MAS,1.129860,2022-12-28,1,2.752476,100,0.7004
4,MCO-WTW,MCO,WTW,1.752769,2022-12-28,-1,-2.668901,102,0.7038


## 3. Price and size each preview

The illustrative budget is 5% of initial capital. Module 07 budgets 5% of current marked equity per entry, limited by available cash, with no open-position count limit.


In [3]:
rows = []
for row in snapshot.itertuples():
    i = test_prices.index.get_loc(row.signal_date)
    if row.horizon <= 1 or i + row.horizon >= len(test_prices):
        rows.append(dict(pair=row.pair, status="no_tradable_remaining_maturity"))
        continue
    entry_date = test_prices.index[i + 1]
    expiry_date = test_prices.index[i + int(row.horizon)]
    instruction = dict(
        pair=row.pair,
        dependent=row.dependent,
        independent=row.independent,
        signal_date=row.signal_date,
        expiry_date=expiry_date,
        direction=row.direction,
    )
    spots, vols, rf, types, prices, deltas = entry_option_terms(
        instruction, entry_date, test_prices, volatility, rates
    )
    budget = cfg.initial_capital * cfg.premium_budget_fraction
    sizing = budgeted_size(
        row.beta,
        spots,
        deltas,
        prices,
        budget,
        cfg.max_hedge_error,
        cfg.slippage_bps,
        cfg.commission_per_contract,
    )
    values = dict(
        pair=row.pair,
        signal_date=row.signal_date,
        entry_date=entry_date,
        expiry_date=expiry_date,
        dependent_type=types[0],
        independent_type=types[1],
        dependent_price=prices[0],
        independent_price=prices[1],
        illustrative_budget=budget,
        status="priced" if sizing else "no_feasible_integer_hedge",
    )
    if sizing:
        values.update(sizing)
    rows.append(values)
option_preview = pd.DataFrame(rows)
option_preview.to_parquet("option_preview.parquet")
display(option_preview)
if option_preview.empty:
    print(
        "No snapshot candidates. Continue to Module 07 for the full daily experiment."
    )


,pair,signal_date,entry_date,expiry_date,dependent_type,independent_type,dependent_price,independent_price,illustrative_budget,status,dependent_contracts,independent_contracts,relative_hedge_error,target_contract_ratio_ind_over_dep,realized_contract_ratio_ind_over_dep
0,ABT-ZTS,2022-12-28,2022-12-29,2023-05-01,put,call,4.452692,11.035881,5000.0,priced,4.0,2.0,0.085803,0.460489,0.5
1,CDW-MSI,2022-12-28,2022-12-29,2023-05-10,call,put,12.370985,12.499207,5000.0,priced,2.0,2.0,0.029171,1.030048,1.0
2,HD-SHW,2022-12-28,2022-12-29,2023-03-29,put,call,13.564644,14.133757,5000.0,no_feasible_integer_hedge,NaN,NaN,NaN,NaN,NaN
3,LEN-MAS,2022-12-28,2022-12-29,2023-05-23,put,call,6.159548,4.133408,5000.0,priced,4.0,6.0,0.042681,1.566875,1.5
4,MCO-WTW,2022-12-28,2022-12-29,2023-05-25,call,put,26.760549,8.584448,5000.0,no_feasible_integer_hedge,NaN,NaN,NaN,NaN,NaN


## 4. Direction and expiry checks

Zero-dividend European options on adjusted synthetic spots remain an explicit model assumption. Costs are research assumptions, not measured option spreads.


In [4]:
assert option_types_from_spread_direction(1) == ("put", "call")
assert option_types_from_spread_direction(-1) == ("call", "put")
assert black_scholes_price(110, 100, 0, 0.03, 0.2, "call") == 10
assert black_scholes_price(90, 100, 0, 0.03, 0.2, "put") == 10
print("Direction and expiry payoff checks passed.")


Direction and expiry payoff checks passed.


## Save module completion

Wait for this confirmation before moving to the next notebook.
